# 2. Data Cleaning & Preparation

### 2.1 Import Libraries & Load Data

In [1]:
import pandas as pd
from pathlib import Path

# Define project root
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Define raw data directory
data_dir = project_root / "data" / "raw" / "Fitabase Data 4.12.16-5.12.16"

# Load datasets
activity_df = pd.read_csv(data_dir / "dailyActivity_merged.csv")
sleep_df = pd.read_csv(data_dir / "sleepDay_merged.csv")

# Display dataset shapes
print("Daily Activity shape:", activity_df.shape)
print("Sleep Data shape:", sleep_df.shape)


Daily Activity shape: (940, 15)
Sleep Data shape: (413, 5)


### 2.2 Initial Data Inspection

In [2]:
# Create a summary of both datasets before cleaning
baseline_summary = pd.DataFrame({
    "Dataset": ["Daily Activity", "Sleep Data"],
    "Rows": [len(activity_df), len(sleep_df)],
    "Columns": [activity_df.shape[1], sleep_df.shape[1]],
    "Missing Values": [
        activity_df.isna().sum().sum(),
        sleep_df.isna().sum().sum()
    ],
    "Exact Duplicates": [
        activity_df.duplicated().sum(),
        sleep_df.duplicated().sum()
    ],
    "Unique Users": [
        activity_df["Id"].nunique(),
        sleep_df["Id"].nunique()
    ]
})

display(baseline_summary)

# Check date column data types
print("\nActivityDate type:", activity_df["ActivityDate"].dtype)
print("SleepDay type:", sleep_df["SleepDay"].dtype)


,Dataset,Rows,Columns,Missing Values,Exact Duplicates,Unique Users
0,Daily Activity,940,15,0,0,33
1,Sleep Data,413,5,0,3,24



ActivityDate type: str
SleepDay type: str


### 2.3 Data Type Conversion

In [3]:
# Create working copies to preserve the original data
activity_clean = activity_df.copy()
sleep_clean = sleep_df.copy()

# Convert date columns to datetime
activity_clean["ActivityDate"] = pd.to_datetime(
    activity_clean["ActivityDate"],
    format="%m/%d/%Y",
    errors="coerce"
)

sleep_clean["SleepDay"] = pd.to_datetime(
    sleep_clean["SleepDay"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

# Verify date conversion
print("ActivityDate type:", activity_clean["ActivityDate"].dtype)
print("SleepDay type:", sleep_clean["SleepDay"].dtype)

print("\nInvalid Activity Dates:", activity_clean["ActivityDate"].isna().sum())
print("Invalid Sleep Dates:", sleep_clean["SleepDay"].isna().sum())

ActivityDate type: datetime64[us]
SleepDay type: datetime64[us]

Invalid Activity Dates: 0
Invalid Sleep Dates: 0


### 2.4 Duplicate Handling

In [5]:
# Review exact duplicate records in Sleep Data
sleep_duplicates = sleep_clean[
    sleep_clean.duplicated(keep=False)
].sort_values(["Id", "SleepDay"])

print("Duplicate rows (including originals):", len(sleep_duplicates))
display(sleep_duplicates)


Duplicate rows (including originals): 6


,Id,SleepDay,TotalSleepRecords,TotalMinutesAsleep,TotalTimeInBed
160,4388161847,2016-05-05,1,471,495
161,4388161847,2016-05-05,1,471,495
222,4702921684,2016-05-07,1,520,543
223,4702921684,2016-05-07,1,520,543
379,8378563200,2016-04-25,1,388,402
380,8378563200,2016-04-25,1,388,402


In [6]:
# Record the number of rows before removing duplicates
rows_before = len(sleep_clean)

# Remove exact duplicate records
sleep_clean = sleep_clean.drop_duplicates().reset_index(drop=True)

# Verify duplicate removal
rows_after = len(sleep_clean)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Duplicates removed:", rows_before - rows_after)
print("Remaining exact duplicates:", sleep_clean.duplicated().sum())


Rows before: 413
Rows after: 410
Duplicates removed: 3
Remaining exact duplicates: 0


### 2.5 Data Validation

In [7]:
# Validate datasets after cleaning
validation_summary = pd.DataFrame({
    "Dataset": ["Daily Activity", "Sleep Data"],
    "Rows": [len(activity_clean), len(sleep_clean)],
    "Unique Users": [
        activity_clean["Id"].nunique(),
        sleep_clean["Id"].nunique()
    ],
    "Missing Values": [
        activity_clean.isna().sum().sum(),
        sleep_clean.isna().sum().sum()
    ],
    "Exact Duplicates": [
        activity_clean.duplicated().sum(),
        sleep_clean.duplicated().sum()
    ],
    "Duplicate User-Date": [
        activity_clean.duplicated(subset=["Id", "ActivityDate"]).sum(),
        sleep_clean.duplicated(subset=["Id", "SleepDay"]).sum()
    ]
})

display(validation_summary)


,Dataset,Rows,Unique Users,Missing Values,Exact Duplicates,Duplicate User-Date
0,Daily Activity,940,33,0,0,0
1,Sleep Data,410,24,0,0,0


In [8]:
# Check negative numeric measurements (excluding user IDs)
activity_numeric = activity_clean.select_dtypes(include="number").drop(columns=["Id"])
sleep_numeric = sleep_clean.select_dtypes(include="number").drop(columns=["Id"])

print("Negative Activity Values:", (activity_numeric < 0).sum().sum())
print("Negative Sleep Values:", (sleep_numeric < 0).sum().sum())

# Check sleep duration consistency
invalid_sleep_duration = (
    sleep_clean["TotalMinutesAsleep"] > sleep_clean["TotalTimeInBed"]
).sum()

print("Sleep duration exceeding time in bed:", invalid_sleep_duration)

# Check non-positive sleep record counts
invalid_sleep_records = (
    sleep_clean["TotalSleepRecords"] <= 0
).sum()

print("Non-positive sleep record counts:", invalid_sleep_records)

# Check activity minutes exceeding 24 hours
activity_minutes = (
    activity_clean["VeryActiveMinutes"]
    + activity_clean["FairlyActiveMinutes"]
    + activity_clean["LightlyActiveMinutes"]
    + activity_clean["SedentaryMinutes"]
)

print("Activity minutes exceeding 1440:", (activity_minutes > 1440).sum())


Negative Activity Values: 0
Negative Sleep Values: 0
Sleep duration exceeding time in bed: 0
Non-positive sleep record counts: 0
Activity minutes exceeding 1440: 0


### 2.6 Export Cleaned Datasets

In [9]:
# Define output directory
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Export cleaned datasets
activity_clean.to_csv(output_dir / "daily_activity_cleaned.csv", index=False)
sleep_clean.to_csv(output_dir / "sleep_cleaned.csv", index=False)

print("Cleaned datasets exported successfully.")


Cleaned datasets exported successfully.


In [10]:
# Reload exported datasets
activity_exported = pd.read_csv(output_dir / "daily_activity_cleaned.csv")
sleep_exported = pd.read_csv(output_dir / "sleep_cleaned.csv")

# Verify exported row counts
print("Exported Activity Rows:", len(activity_exported))
print("Exported Sleep Rows:", len(sleep_exported))


Exported Activity Rows: 940
Exported Sleep Rows: 410
